# Project: V7P3R Chess Variant AI — Diagnostic & Strategy Notebook
**Objective:** *Analyze failure modes of BlackBoxBreakerCNN and implement Hindsight Backtracking.*

### 🛠️ Section 1: Environment & Architecture Setup
1. Load the 50,000-puzzle test dataset JSON.
2. Instantiate BlackBoxBreakerCNN and load the chess_model_best.pth weights.
3. Verify tensor dimensions across the pipeline.

### 🔍 Section 2: Error Classification & Distribution (The "Why")
- Row/Column/Block Violation Matrix: Code to parse the failed boards and count exactly which constraints are violated most frequently.
- Confidence vs. Correctness Plot: A scatter plot matching the model's confidence for a selection against whether the digit actually matches the ground truth solution.
- The "Blunder Step" Distribution: A histogram showing exactly at which step (Step 1, Step 2, etc.) the first logical mistake typically occurs across 100 failed puzzles.

### 🔄 Section 3: The Hindsight Wrapper (Backtracking Integration)
1. Define the state tracking mechanism (Move History).
2. Implement the confidence threshold parameter ($\tau$) to trigger a roll-back when a downstream confidence drop is detected.
3. Comparative benchmark: Run the new wrapper over the same test batch to chart the accuracy recovery curve.

#### 🗂️ Designing the Move History Tracker
To build the Hindsight Wrapper in Section 3, we need to track our journey through the decision tree so we can step backward when confidence plunges.  
A natural way to store this history is using a standard Python list as a Stack data structure, utilizing append() to push moves and pop() to undo them.

### Appendix: Utilities & Debugging Aids
- ML pipeline integrity checks and ground truth verification code snippets.

In [1]:
# Load the test dataset JSON
DATASET_ID = "202606171718"  # Update this with the actual dataset ID used in data_preparer.py
TARGET_TEST_DATASET = f"data/encoded/split/chess_puzzle_training_dataset_{DATASET_ID}_test.json"
LOG_PATH = f"analysis/logs/results_chess_puzzle_training_dataset_{DATASET_ID}_test.log"
JSON_FAILURES_PATH = f"analysis/logs/failed_puzzles_analysis_chess_puzzle_training_dataset_{DATASET_ID}_test.json"

In [4]:
def count_chess_puzzles(filepath):
    count = 0
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  # Skip accidental empty lines
                count += 1
    return count

# This will run smoothly even if the file is 50GB+
total_puzzles = count_chess_puzzles(TARGET_TEST_DATASET)
print(f"Total puzzles in dataset: {total_puzzles:,}")


Total puzzles in dataset: 761,505


In [ ]:
# Section 1: Environment & Configuration
import json
import torch
import torch.nn.functional as F
from train import BlackBoxBreakerCNN

# 1. Load the split test dataset JSON
TEST_DATA_PATH = TARGET_TEST_DATASET
try:
    with open(TEST_DATA_PATH, "r") as f:
        test_json = json.load(f)
    print(f"📦 Successfully loaded test dataset with {len(test_json)} puzzles.")
except FileNotFoundError:
    print(f"❌ Error: Failed to load test dataset from {TEST_DATA_PATH}. Please ensure the file exists and the path is correct.")
    test_json = []

# 2. Instantiate the network and load trained weights
model = BlackBoxBreakerCNN()
model.load_state_dict(torch.load("models/chess_model_best.pth", map_location=torch.device('cpu')))  # Load to CPU for testing
model.eval()
print("🤖 Model architecture loaded and weights initialized.")

# 3. Verify tensor dimensions across the pipeline
# Grab a single raw sample from our loaded JSON
sample_entry = test_json[0]
raw_puzzle = torch.tensor(sample_entry['puzzle'], dtype=torch.float32)
print(f"\n📐 1. Shape from JSON dataset: {raw_puzzle.shape}")

# Add the batch dimension for a single inference batch size of 1
input_tensor = raw_puzzle.unsqueeze(0)
print(f"📐 2. Final 4D input shape ready for model: {input_tensor.shape}")

# Run a forward pass to check output shape consistency
with torch.no_grad():
    output_tensor = model(input_tensor)
print(f"📐 3. Model output prediction shape: {output_tensor.shape}")

In [ ]:
# Section 2: Error Classification & Distribution
import json
import matplotlib.pyplot as plt

# Load the compiled failure data
try:
    with open(JSON_FAILURES_PATH, "r") as f:
        failed_data = json.load(f)
    print(f"📊 Successfully loaded data with {len(failed_data)} total entries.")
except FileNotFoundError:
    print(f"❌ Error: Failed to load data from {JSON_FAILURES_PATH}.")
    failed_data = []

# Aggregate metrics strictly across actual failures
total_violations = {"row": 0, "col": 0, "block": 0}
actual_failures_count = 0
alternative_solutions_count = 0

for entry in failed_data:
    # Skip entries that the model solved matching the ground truth
    if entry["is_correct"]:
        continue
        
    actual_failures_count += 1
    
    # Calculate sum of violations for this specific failed board
    puzzle_violations = entry["violations"]["row"] + entry["violations"]["col"] + entry["violations"]["block"]
    
    if puzzle_violations == 0:
        # Perfectly valid board state but didn't match the recorded solution array
        alternative_solutions_count += 1
    else:
        # True logical breakdown
        total_violations["row"] += entry["violations"]["row"]
        total_violations["col"] += entry["violations"]["col"]
        total_violations["block"] += entry["violations"]["block"]

print(f"\n================ Forensic Summary ================")
print(f"❌ Total Solution Mismatches: {actual_failures_count}")
print(f"👻 Valid Alternative Solutions Found: {alternative_solutions_count} ({ (alternative_solutions_count/max(1, actual_failures_count))*100:.2f}% of misses)")
print(f"💥 True Constraint Breakdowns: {actual_failures_count - alternative_solutions_count}")
print(f"==================================================")

print("\n📊 True Infraction Distribution Across Failed Puzzles:")
violation_count = 0
for constraint, count in total_violations.items():
    violation_count += count
    print(f"   + {constraint.capitalize()} Violations: {count}")
print(f"    --------------------\n   = Total Structural Violations: {violation_count}")

if violation_count > 0:
    # Plot the distribution of true logic bugs
    plt.figure(figsize=(8, 5))
    plt.bar(total_violations.keys(), total_violations.values(), color=['#FF6B6B', '#4D96FF', '#6BCB77'])
    plt.title("Distribution of True Chess Constraint Violations")
    plt.ylabel("Total Infractions Detected")
    plt.xlabel("Constraint Type")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
# Pipeline Testing diagnostics

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import datetime

# 1. Load your newly generated failure logs from the extreme dataset
with open(JSON_FAILURES_PATH, 'r') as f:
    failed_puzzles = json.load(f)

all_ground_truths = []
all_predictions = []

print(f"🔮 Parsing step histories across {len(failed_puzzles)} failed instances...")

# 2. Extract step-by-step decisions
for puzzle in failed_puzzles:
    solution_matrix = np.array(puzzle['ground_truth'])
    
    for step in puzzle['steps']:
        r, c = step['cell']
        predicted_digit = step['digit']
        true_digit = solution_matrix[r, c]
        
        all_ground_truths.append(true_digit)
        all_predictions.append(predicted_digit)

# Convert to clean numpy arrays for processing (targeting digits 1-9)
y_true = np.array(all_ground_truths)
y_pred = np.array(all_predictions)

# 3. Print out your Core Data Science Metrics (Uncorrupted, calculates perfectly!)
print("\n===== 📊 CLASS-SPECIFIC PERFORMANCE REPORT =====")
target_names = [f"Digit {i}" for i in range(1, 10)]
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

# 4. Generate the base Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(1, 10)))

# 5. FORENSIC MASKING STEP: Zero out the diagonal to reveal hidden error structures
cm_errors_only = cm.copy()
np.fill_diagonal(cm_errors_only, 0)

# Create a text annotation mask that replaces zeros on the diagonal with a hyphen "-"
annot_labels = np.where(
    np.eye(cm.shape[0], dtype=bool), 
    "-", 
    cm_errors_only.astype(str)
)

figure_datestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# 6. Render the Error-Isolated Heatmap
plt.figure(figsize=(11, 9))
sns.heatmap(
    cm_errors_only, 
    annot=annot_labels, # Place hyphens on the true diagonal, raw error counts everywhere else
    fmt='s',            # Use string formatting to accommodate the hyphens
    cmap='Oranges',     # Switch to Oranges/Reds to distinctively mark a fault map
    xticklabels=range(1, 10), 
    yticklabels=range(1, 10),
    cbar_kws={'label': 'Volume of Active Mispredictions'}
)
plt.title('Chess AI Isolated Error Matrix (Correct Guesses Masked)', fontsize=14, pad=15)
plt.xlabel('Predicted Digit (Model\'s Decision)', fontsize=12)
plt.ylabel('Ground Truth Digit (Actual Solution)', fontsize=12)
plt.tight_layout()
plt.savefig(f"analysis/test_runs/test_run_{DATASET_ID}_{figure_datestamp}.png", dpi=300)
plt.show()

# Save plot to analysis/test_runs
